In [20]:
import json
import random

In [21]:
# Function to sample a specific number of tokens
def sample_tokens(data, token_limit):
    selected = []
    total_tokens = 0
    for item in data:
        if total_tokens + item['len'] <= token_limit:
            total_tokens += item['len']
            selected.append({k: v for k, v in item.items() if k != 'len'})
        else:
            # Add a truncated version of the last item
            remaining_tokens = token_limit - total_tokens
            truncated_item = {k: v for k, v in item.items() if k != 'len'}
            truncated_item['output'] = truncated_item['output'][:remaining_tokens]
            selected.append(truncated_item)
            break
    return selected

# Load the JSON data
# with open('/mbz/users/liyuan/LLaMA-Factory/data/openmathinstruct2_1M_len.json', 'r') as f:

with open('/mbz/users/liyuan/LLaMA-Factory/data/Infinity-Instruct_0625_len.json', 'r') as f:
    structured_data = json.load(f)
    
    
# with open('/mbz/users/liyuan/LLaMA-Factory/data/opencoder-sft_len.json', 'r') as f:
#     structured_data = json.load(f) 

item_num = 323_000
if len(structured_data) >= item_num:
    sampled_data = random.sample(structured_data, item_num)
else:
    raise ValueError(f"The dataset contains fewer than {item_num} items.")

# Calculate the total tokens
total_tokens = sum(item['len'] for item in sampled_data)

print(f"Total tokens for the sampled {item_num} items: {total_tokens}")

Total tokens for the sampled 323000 items: 174044404


In [22]:
base_token = 200_000_000
experiment_name = "exp2"
validation_size = 1000

token_limits = {
    "0.125": int(base_token * 0.125),
    "0.25": int(base_token * 0.25),
    "0.375": int(base_token * 0.375),
    "0.5": int(base_token * 0.5),
    "0.625": int(base_token * 0.625),
    "0.75": int(base_token * 0.75)
}

In [8]:
domain = "math"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/openmathinstruct2_1M_len.json"
val_output_path = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{base_token}_{domain}_val.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)
    
validation_set = random.sample(structured_data, validation_size)
    
validation_set_ids = set(map(lambda x: json.dumps(x, sort_keys=True), validation_set))
reduced_data = []
for item in structured_data:
    if json.dumps(item, sort_keys=True) not in validation_set_ids:
        reduced_data.append(item)

# 4. Overwrite the original dataset file with the reduced data
with open(original_dataset_path, 'w') as f:
    json.dump(reduced_data, f, indent=4)

# 5. Save the validation set (dropping 'len' if desired)
filtered_validation = [{k: v for k, v in item.items() if k != 'len'} for item in validation_set]
with open(val_output_path, 'w') as f:
    json.dump(filtered_validation, f, indent=4)
print(f"Validation set created with {len(validation_set)} items.")

Validation set created with 1000 items.


In [9]:
domain = "code"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/opencoder-sft_len.json"
val_output_path = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{base_token}_{domain}_val.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)
    
validation_set = random.sample(structured_data, validation_size)

validation_set_ids = set(map(lambda x: json.dumps(x, sort_keys=True), validation_set))
reduced_data = []
for item in structured_data:
    if json.dumps(item, sort_keys=True) not in validation_set_ids:
        reduced_data.append(item)

# 4. Overwrite the original dataset file with the reduced data
with open(original_dataset_path, 'w') as f:
    json.dump(reduced_data, f, indent=4)

# 5. Save the validation set (dropping 'len' if desired)
filtered_validation = [{k: v for k, v in item.items() if k != 'len'} for item in validation_set]
with open(val_output_path, 'w') as f:
    json.dump(filtered_validation, f, indent=4)
print(f"Validation set created with {len(validation_set)} items.")

Validation set created with 1000 items.


In [7]:
domain = "instr"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/Infinity-Instruct_0625_len.json"
val_output_path = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{base_token}_{domain}_val.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)
    
validation_set = random.sample(structured_data, validation_size)

validation_set_ids = set(map(lambda x: json.dumps(x, sort_keys=True), validation_set))
reduced_data = []
for item in structured_data:
    if json.dumps(item, sort_keys=True) not in validation_set_ids:
        reduced_data.append(item)

# 4. Overwrite the original dataset file with the reduced data
with open(original_dataset_path, 'w') as f:
    json.dump(reduced_data, f, indent=4)

# 5. Save the validation set (dropping 'len' if desired)
filtered_validation = [{k: v for k, v in item.items() if k != 'len'} for item in validation_set]
with open(val_output_path, 'w') as f:
    json.dump(filtered_validation, f, indent=4)
print(f"Validation set created with {len(validation_set)} items.")

Validation set created with 1000 items.


## Start sample training data

In [23]:
domain = "code"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/opencoder-sft_len.json"
val_output_path = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{base_token}_{domain}_val.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

# 1. Load and shuffle the data
with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)

random.shuffle(structured_data)

# Define token limits

# 6. Perform sampling for each token limit
all_selected_items = []
for name, limit in token_limits.items():
    sampled_items = sample_tokens(structured_data, limit)
    all_selected_items.extend(sampled_items)

    # Save each sampled subset
    subset_path = f"{save_dir}/{base_token}_{domain}_{name}.json"
    with open(subset_path, 'w') as f:
        json.dump(sampled_items, f, indent=4)

    print(f"Sampled {len(sampled_items)} items for {name} with token limit {limit}.")

Sampled 54088 items for 0.125 with token limit 25000000.
Sampled 108121 items for 0.25 with token limit 50000000.
Sampled 162590 items for 0.375 with token limit 75000000.
Sampled 216860 items for 0.5 with token limit 100000000.
Sampled 271230 items for 0.625 with token limit 125000000.
Sampled 325621 items for 0.75 with token limit 150000000.


In [24]:
domain = "instr"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/Infinity-Instruct_0625_len.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

# 1. Load and shuffle the data
with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)

random.shuffle(structured_data)

all_selected_items = []
for name, limit in token_limits.items():
    sampled_items = sample_tokens(structured_data, limit)
    all_selected_items.extend(sampled_items)

    # Save each sampled subset
    subset_path = f"{save_dir}/{base_token}_{domain}_{name}.json"
    with open(subset_path, 'w') as f:
        json.dump(sampled_items, f, indent=4)

    print(f"Sampled {len(sampled_items)} items for {name} with token limit {limit}.")

Sampled 46572 items for 0.125 with token limit 25000000.
Sampled 92830 items for 0.25 with token limit 50000000.
Sampled 139301 items for 0.375 with token limit 75000000.
Sampled 185512 items for 0.5 with token limit 100000000.
Sampled 232035 items for 0.625 with token limit 125000000.
Sampled 278624 items for 0.75 with token limit 150000000.


In [25]:
domain = "math"

original_dataset_path = "/mbz/users/liyuan/LLaMA-Factory/data/openmathinstruct2_1M_len.json"
save_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"

# 1. Load and shuffle the data
with open(original_dataset_path, 'r') as f:
    structured_data = json.load(f)

random.shuffle(structured_data)

all_selected_items = []
for name, limit in token_limits.items():
    sampled_items = sample_tokens(structured_data, limit)
    all_selected_items.extend(sampled_items)

    # Save each sampled subset
    subset_path = f"{save_dir}/{base_token}_{domain}_{name}.json"
    with open(subset_path, 'w') as f:
        json.dump(sampled_items, f, indent=4)

    print(f"Sampled {len(sampled_items)} items for {name} with token limit {limit}.")

Sampled 56643 items for 0.125 with token limit 25000000.
Sampled 113084 items for 0.25 with token limit 50000000.
Sampled 169488 items for 0.375 with token limit 75000000.
Sampled 225612 items for 0.5 with token limit 100000000.
Sampled 282313 items for 0.625 with token limit 125000000.
Sampled 338823 items for 0.75 with token limit 150000000.


## Store data name into data_info

In [26]:
dataset_info_path = "/mbz/users/liyuan/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

for domain in ["instr", "math", "code"]:
    # dataset_name = f"{base_token}_{domain}_val"
    # output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
    # dataset_info[dataset_name] = {
    #     "file_name": output_path
    # }
    for size in token_limits.keys():
        dataset_name = f"{base_token}_{domain}_{size}"
        output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
        dataset_info[dataset_name] = {
            "file_name": output_path
        }
        
    with open(dataset_info_path, "w") as f:
        json.dump(dataset_info, f, indent=2)

In [8]:
dataset_info_path = "/mbz/users/liyuan/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

combined_data = []

for data_source in [f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{base_token}_code_val.json", f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{base_token}_math_val.json", f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{base_token}_instr_val.json"]:
    with open(data_source, "r") as read_file:
        data = json.load(read_file)
    combined_data.extend(data)
    
dataset_name = f"{base_token}_{experiment_name}_val"
with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{dataset_name}.json", "w") as f:
    json.dump(combined_data, f, indent=2)
    

output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
dataset_info[dataset_name] = {
    "file_name": output_path
}

with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

In [10]:
with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{dataset_name}.json", "r") as f:
    data = json.load(f)
    print(len(data))

3000
